In [50]:
from dataclasses import dataclass
import pickle
import gzip
import random
import numpy as np
from PIL import Image, ImageOps
from time import time
import matplotlib.pyplot as plt

In [51]:
@dataclass
class Network:
    num_layers: int
    biases: list
    weights: list

In [52]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_prime(z):
    return sigmoid(z) * (1 - sigmoid(z))

In [53]:
def init_network(layers):
    num_layers = len(layers)

    biases = [np.random.randn(y, 1) for y in layers[1:]]

    weights = [np.random.randn(y, x)
               for x, y in zip(layers[:-1], layers[1:])]

    return Network(num_layers, biases, weights)

In [54]:
def feedforward(nn, a):

    for b, w in zip(nn.biases, nn.weights):
        a = sigmoid(np.dot(w, a) + b)

    return a

In [55]:
def evaluate(nn, test_data):

    test_results = [
        (np.argmax(feedforward(nn, x)), y)
        for (x, y) in test_data
    ]

    return sum(int(x == y)
               for (x, y) in test_results)

In [56]:
def cost_derivative(output_activations, y):

    return output_activations - y

In [57]:
def backprop_batch(nn, X, Y):

    activations = [X]
    zs = []

    A = X

    # Feedforward
    for b, w in zip(nn.biases, nn.weights):

        Z = np.dot(w, A) + b

        zs.append(Z)

        A = sigmoid(Z)

        activations.append(A)

    m = X.shape[1]

    nabla_b = [np.zeros(b.shape) for b in nn.biases]
    nabla_w = [np.zeros(w.shape) for w in nn.weights]

    # Output layer
    delta = (
        activations[-1] - Y
    ) * sigmoid_prime(zs[-1])

    nabla_b[-1] = np.sum(
        delta,
        axis=1,
        keepdims=True
    ) / m

    nabla_w[-1] = np.dot(
        delta,
        activations[-2].T
    ) / m

    # Hidden layers
    for l in range(2, nn.num_layers):

        delta = np.dot(
            nn.weights[-l + 1].T,
            delta
        ) * sigmoid_prime(zs[-l])

        nabla_b[-l] = np.sum(
            delta,
            axis=1,
            keepdims=True
        ) / m

        nabla_w[-l] = np.dot(
            delta,
            activations[-l - 1].T
        ) / m

    return nabla_b, nabla_w

In [58]:
def stochastic_gradient_descent(
    nn,
    mini_batch,
    eta
):

    # Create batch matrices

    X = np.hstack([
        x for x, y in mini_batch
    ])

    Y = np.hstack([
        y for x, y in mini_batch
    ])

    nabla_b, nabla_w = backprop_batch(
        nn,
        X,
        Y
    )

    nn.weights = [
        w - eta * nw
        for w, nw
        in zip(nn.weights, nabla_w)
    ]

    nn.biases = [
        b - eta * nb
        for b, nb
        in zip(nn.biases, nabla_b)
    ]

In [59]:
def learn(nn,
          training_data,
          epochs,
          mini_batch_size,
          learning_rate,
          test_data=None):

    n = len(training_data)

    for j in range(epochs):

        random.shuffle(training_data)

        mini_batches = [

            training_data[k:k + mini_batch_size]

            for k in range(
                0,
                n,
                mini_batch_size
            )
        ]

        for mini_batch in mini_batches:

            stochastic_gradient_descent(
                nn,
                mini_batch,
                learning_rate
            )

        if test_data:

            accuracy = (
                100.0 *
                evaluate(nn, test_data)
                / len(test_data)
            )

            print(
                f"Epoch {j+1}: "
                f"accuracy {accuracy:.2f}%"
            )

In [60]:
def one_hot_encode(j):

    e = np.zeros((10, 1))

    e[j] = 1.0

    return e

In [61]:
def load_data():

    f = gzip.open(
        'mnist.pkl.gz',
        'rb'
    )

    training_data, validation_data, test_data = \
        pickle.load(f, encoding="bytes")

    f.close()

    return (
        training_data,
        validation_data,
        test_data
    )

In [62]:
def load_data_wrapper():

    tr_d, va_d, te_d = load_data()

    training_inputs = [
        np.reshape(x, (784, 1))
        for x in tr_d[0]
    ]

    training_results = [
        one_hot_encode(y)
        for y in tr_d[1]
    ]

    training_data = list(
        zip(
            training_inputs,
            training_results
        )
    )

    validation_inputs = [
        np.reshape(x, (784, 1))
        for x in va_d[0]
    ]

    validation_data = list(
        zip(
            validation_inputs,
            va_d[1]
        )
    )

    test_inputs = [
        np.reshape(x, (784, 1))
        for x in te_d[0]
    ]

    test_data = list(
        zip(
            test_inputs,
            te_d[1]
        )
    )

    return (
        training_data,
        validation_data,
        test_data
    )

In [63]:
training_data, validation_data, test_data = \
    load_data_wrapper()

nn = init_network([784, 30, 10])

epochs = 15
mini_batch_size = 10
learning_rate = 3.0

start = time()

learn(
    nn,
    training_data,
    epochs,
    mini_batch_size,
    learning_rate,
    test_data
)

end = time()

print(
    f"\nTraining Time: "
    f"{end-start:.2f} seconds"
)

accuracy = (
    100 *
    evaluate(nn, validation_data)
    / len(validation_data)
)

print(
    f"Validation Accuracy: "
    f"{accuracy:.2f}%"
)

Epoch 1: accuracy 90.80%
Epoch 2: accuracy 92.44%
Epoch 3: accuracy 93.34%
Epoch 4: accuracy 93.68%
Epoch 5: accuracy 94.36%
Epoch 6: accuracy 93.95%
Epoch 7: accuracy 94.30%
Epoch 8: accuracy 94.43%
Epoch 9: accuracy 94.66%
Epoch 10: accuracy 94.57%
Epoch 11: accuracy 94.86%
Epoch 12: accuracy 94.59%
Epoch 13: accuracy 94.97%
Epoch 14: accuracy 94.82%
Epoch 15: accuracy 95.06%

Training Time: 29.94 seconds
Validation Accuracy: 95.22%


In [64]:
incorrect = []

for x, actual in test_data:

    predicted = np.argmax(
        feedforward(nn, x)
    )

    if predicted != actual:
        incorrect.append(
            (x, actual, predicted)
        )

print(
    "Incorrect Predictions:",
    len(incorrect)
)

Incorrect Predictions: 494


In [65]:
mini_batch = training_data[:10]

X = np.hstack([x for x, y in mini_batch])
Y = np.hstack([y for x, y in mini_batch])

print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: (784, 10)
Y shape: (10, 10)
